# Oscillation-effect analysis: correlation, z-score, and cross-correlation

Analyzes the four oscillator scenarios in
`simulation_data/synthetic_network/oscillator_scenarios/`
(`in_phase`, `out_of_phase`, `twin_dephasing`, `async_division`), each with 3
replicate simulations of 6000 twin pairs, measured every 1h from 0-48h after
division.

For every scenario / replicate this notebook computes, using only functions
from
[`correlation_functions.py`](../package/twinfer/inference/correlation_functions.py):

- **Twin-pair correlation** (`calculate_twin_random_pair_correlations`) — Spearman
  correlation of Δgene_i vs Δgene_j between twins, at every timepoint.
- **Z-score** (`generate_random_shuffle` + `get_correlations`) — how far the twin
  correlation sits from a random-pair permutation null, at every timepoint.
- **Directed cross-correlation** (`get_cross_correlations`, plus a permutation
  p-value built from the same private rank kernels `identify_actual_directed_edges`
  uses internally) — correlation between gene_i in one twin at time t1 and gene_j
  in the other twin at time t2, for t1 &isin; {1, 5, 10, 15, 20}h swept against every
  available t2.
- **Gene expression trajectories** — mean mRNA and protein per gene over time,
  averaged across the 3 replicates.

**Bug fix applied to the shared library:** `generate_random_shuffle`'s null-generation
kernel (`_half_split_diff_null_kernel`) had a hardcoded acceptance band
(`1470 <= n_used <= 1500`) that silently discarded *every* shuffle — and returned
an all-`NaN` null, hence `NaN` z-scores — for any dataset that doesn't have almost
exactly 1500 twin pairs. This dataset has 6000 twin pairs per timepoint, so it hit
that bug on every call. The fix generalizes the band to ±2% of the dynamically
computed target (exactly reproducing the old behavior at 1500 pairs). See
`correlation_functions.py` around `_half_split_diff_null_kernel` for the fix.

**Runtime:** with `QUICK_TEST = False` (full `N_SHUFFLES = 10000`, full 1h-resolution
time sweep), the two sweep cells below are the expensive part of this notebook and
took on the order of 1-2 hours combined in testing (256-core node). Set
`QUICK_TEST = True` first to sanity-check the whole notebook end-to-end in under a
minute before committing to the full run.


In [ ]:
from twinfer.utils.paths import get_data_root as _twinfer_get_data_root
TWINFER_PROJECT_ROOT = _twinfer_get_data_root().parent  # [2026-09-30 added: replaces hardcoded project-root paths (/home/gzu5140/TwINFER_KA, /gpfs/projects/b1255/hzhang/TwINFER_KA, old Keerthana_b1042 tree)]
import sys
from pathlib import Path
from itertools import combinations, permutations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tqdm.auto import tqdm

PACKAGE_ROOT = f'{TWINFER_PROJECT_ROOT}/code/TwINFER/package'
if PACKAGE_ROOT not in sys.path:
    pass
    # [2026-09-30 commented out: modules are imported via dotted package paths (env.sh puts clean_code and clean_code/package on PYTHONPATH)]
    # sys.path.insert(0, PACKAGE_ROOT)

from twinfer.inference import correlation_functions as cf

%matplotlib inline


In [ ]:
from twinfer.utils.paths import get_data_root as _twinfer_get_data_root
TWINFER_PROJECT_ROOT = _twinfer_get_data_root().parent  # [2026-09-30 added: replaces hardcoded project-root paths (/home/gzu5140/TwINFER_KA, /gpfs/projects/b1255/hzhang/TwINFER_KA, old Keerthana_b1042 tree)]
# ---- Config -----------------------------------------------------------
BASE_DIR = Path(f'{TWINFER_PROJECT_ROOT}/simulation_data/synthetic_network/oscillator_scenarios_redo')
SCENARIOS = ["in_phase", "out_of_phase"]

T1_ANCHORS = [5]     # hours; cross-correlation reference (source) timepoints
HEADLINE_T1, HEADLINE_T2 = 5, 15    # the single (t1, t2) pair highlighted as heatmaps

BASE_SEED = 101010
N_CORES = 32                        # numba thread count for the permutation kernels

RESULTS_DIR = Path(f'{TWINFER_PROJECT_ROOT}/clean_data/notebooks/real_data_analysis/oscillation_effect_results_redo')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Flip to True first to sanity-check every cell below runs end-to-end in ~seconds,
# then set back to False for the full-precision run described in the intro.
QUICK_TEST = False

N_SHUFFLES = 300 if QUICK_TEST else 10000
T2_SWEEP_STEP = 5 if QUICK_TEST else 1   # hours between swept t2 values

import numba
numba.set_num_threads(min(N_CORES, numba.config.NUMBA_NUM_THREADS))


In [ ]:
# ---- Discover & load the per-replicate simulation files ----------------
def discover_replicate_files(scenario):
    '''Main post-division measurement CSVs for a scenario, keyed by replicate label.

    Excludes the 'simulation_before_division_*' and 'division_times_*' files that
    sit alongside them in the same folder (steady-state / division-timing metadata,
    not the twin measurement table used here).
    '''
    folder = BASE_DIR / scenario
    files = sorted(
        p for p in folder.glob("df_rows_*.csv")
        if not p.name.startswith("simulation_before_division")
    )
    reps = {}
    for p in files:
        rep_label = p.stem.split(f"{scenario}_")[-1].split("_")[:2]
        rep_label = "_".join(rep_label)  # e.g. "rep_0"
        reps[rep_label] = p
    return dict(sorted(reps.items()))


FILES = {scenario: discover_replicate_files(scenario) for scenario in SCENARIOS}
for scenario, reps in FILES.items():
    print(scenario, "->", list(reps.keys()))


In [ ]:
DATA = {
    scenario: {rep_label: pd.read_csv(path) for rep_label, path in reps.items()}
    for scenario, reps in FILES.items()
}

# Gene list: detect from columns (e.g. 'gene_1_mRNA' -> 'gene_1'), sorted numerically.
_first_df = next(iter(next(iter(DATA.values())).values()))
GENE_LIST_RAW = sorted({c.rsplit("_", 1)[0] for c in _first_df.columns if c.endswith("_mRNA")})
GENE_LIST, _ = cf.sort_genes_numerically(GENE_LIST_RAW)
print("genes:", GENE_LIST)

ALL_TIME_STEPS = sorted(_first_df["time_step"].unique().tolist())
for scenario, reps in DATA.items():
    for rep_label, df in reps.items():
        ts = sorted(df["time_step"].unique().tolist())
        assert ts == ALL_TIME_STEPS, f"{scenario}/{rep_label} has a different time grid: {ts}"
print(f"time_step grid: {ALL_TIME_STEPS[0]}..{ALL_TIME_STEPS[-1]}h, "
      f"{len(ALL_TIME_STEPS)} points, all files agree")


## 1. Gene expression trajectories (mRNA & protein), averaged across replicates

For each scenario/gene/metric, the per-cell mean is taken within each replicate
at every timepoint, then the 3 replicate-level means are averaged (shaded band =
SD across the 3 replicates).


In [ ]:
expr_records = []
for scenario, reps in DATA.items():
    for rep_label, df in reps.items():
        for gene in GENE_LIST:
            for metric in ["mRNA", "protein"]:
                rep_mean = df.groupby("time_step")[f"{gene}_{metric}"].mean()
                for t, val in rep_mean.items():
                    expr_records.append(dict(scenario=scenario, rep=rep_label, gene=gene,
                                              metric=metric, time_step=t, rep_mean=val))

expr_df = pd.DataFrame(expr_records)
expr_summary = (
    expr_df.groupby(["scenario", "gene", "metric", "time_step"])["rep_mean"]
    .agg(mean="mean", sd="std", n_reps="count")
    .reset_index()
)
expr_summary.to_csv(RESULTS_DIR / "expression_trajectories.csv", index=False)
expr_summary.head()


In [ ]:
_scenario_colors = dict(zip(SCENARIOS, plt.cm.tab10.colors))

for metric in ["mRNA", "protein"]:
    fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
    for ax, gene in zip(axes.flat, GENE_LIST):
        for scenario in SCENARIOS:
            sub = expr_summary.query(
                "metric == @metric and gene == @gene and scenario == @scenario"
            ).sort_values("time_step")
            color = _scenario_colors[scenario]
            ax.plot(sub["time_step"], sub["mean"], label=scenario, color=color, linewidth=1.75)
            ax.fill_between(sub["time_step"], sub["mean"] - sub["sd"], sub["mean"] + sub["sd"],
                             color=color, alpha=0.2, linewidth=0)
        ax.set_ylim(0, 5)
        ax.set_title(gene)
        ax.set_xlabel("time after division (h)")
        ax.set_ylabel(f"mean {metric} per cell")
    axes.flat[0].legend(fontsize=9)
    fig.suptitle(f"Gene expression ({metric}) averaged across 3 replicates ± SD across replicates")
    plt.tight_layout()
    plt.show()


## 2. Correlation / z-score / cross-correlation helpers

Both helpers below are thin orchestration around `correlation_functions.py`'s own
building blocks. They deliberately avoid the module's higher-level classification
functions (`differentiate_single_state_reg_and_multiple_states`,
`identify_actual_directed_edges`, `check_gene_gene_correlation_threshold`) for the
*sweep* below: those call `plot_qq_distribution` unconditionally on every
significant pair regardless of `verbose`, which is fine for a single (t1, t2) call
but would render thousands of figures across a multi-timepoint sweep. The same
underlying null-generation kernels (`generate_random_shuffle`,
`_rank_center_and_sumsq`, `_rank_permutation_null_kernel`) are used directly instead
&mdash; no statistics are reimplemented, just the plotting/printing side effects are
skipped.


In [ ]:
def compute_twin_correlation_and_zscore(df_t, gene_list, n_shuffles, base_seed):
    '''Twin-pair correlation matrix + z-score of each pair against the random-pair null.

    Returns
    -------
    twin_corr : pd.DataFrame  (gene x gene, Spearman corr of twin-pair deltas)
    random_point_corr : pd.DataFrame  (gene x gene, single random-pairing estimate)
    z_score : pd.DataFrame  (gene x gene, (twin_corr - null_mean) / null_std; NaN on diagonal)
    '''
    twin_corr, random_point_corr = cf.calculate_twin_random_pair_correlations(df_t, df_t, gene_list)
    null_dict = cf.generate_random_shuffle(df_t, gene_list=gene_list, n_shuffles=n_shuffles,
                                            random_state=base_seed)

    z_score = pd.DataFrame(np.nan, index=gene_list, columns=gene_list, dtype=float)
    for gi, gj in combinations(gene_list, 2):
        null_vals = cf.get_correlations(null_dict, gi, gj)
        mu, sigma = np.mean(null_vals), np.std(null_vals)
        z = np.nan if sigma == 0 else (twin_corr.loc[gi, gj] - mu) / sigma
        z_score.loc[gi, gj] = z
        z_score.loc[gj, gi] = z
    return twin_corr, random_point_corr, z_score


def compute_directed_cross_correlation(df_t1, df_t2, gene_list, n_shuffles, base_seed):
    '''Directed t1->t2 cross-correlation matrix + a permutation p-value per pair.

    rep_0 (one twin) is read at t1, rep_1 (the other twin, same clone_id) at t2,
    matching get_cross_correlations' own convention. The p-value uses the same
    rank-permutation null kernel identify_actual_directed_edges builds internally,
    called directly here to avoid that function's unconditional QQ-plotting.

    Returns
    -------
    raw_matrix : pd.DataFrame  (gene x gene, directed Spearman corr, incl. diagonal)
    p_value : pd.DataFrame  (gene x gene, two-sided permutation p-value)
    '''
    rep_0_t1, _ = cf.split_twin_pairs(df_t1)
    _, rep_1_t2 = cf.split_twin_pairs(df_t2)

    gene_pairs = list(permutations(gene_list, 2))
    raw_matrix = cf.get_cross_correlations(rep_0_t1, rep_1_t2, gene_pairs, type_comparison="twin")

    rep_0_t1 = rep_0_t1.sort_values("clone_id").reset_index(drop=True)
    rep_1_t2 = rep_1_t2.sort_values("clone_id").reset_index(drop=True)
    gene_matrix_t1 = np.array([rep_0_t1[f"{g}_mRNA"].values for g in gene_list], dtype=np.float64)
    gene_matrix_t2 = np.array([rep_1_t2[f"{g}_mRNA"].values for g in gene_list], dtype=np.float64)
    cf._assert_no_nan(gene_matrix_t1, "compute_directed_cross_correlation (t1)")
    cf._assert_no_nan(gene_matrix_t2, "compute_directed_cross_correlation (t2)")

    Rc1, s2_1 = cf._rank_center_and_sumsq(gene_matrix_t1)
    Rc2, s2_2 = cf._rank_center_and_sumsq(gene_matrix_t2)
    denom = np.sqrt(np.outer(s2_1, s2_2))
    denom[denom == 0] = np.nan

    gene_to_idx = {g: i for i, g in enumerate(gene_list)}
    all_pairs = [(gi, gj) for gi in gene_list for gj in gene_list]
    pair_i = np.array([gene_to_idx[a] for a, b in all_pairs], dtype=np.int64)
    pair_j = np.array([gene_to_idx[b] for a, b in all_pairs], dtype=np.int64)

    seeds = cf._spawn_independent_seeds(base_seed, n_shuffles)
    null_matrix = cf._rank_permutation_null_kernel(Rc1, Rc2, denom, seeds, pair_i, pair_j)

    p_value = pd.DataFrame(np.nan, index=gene_list, columns=gene_list, dtype=float)
    for k, (gi, gj) in enumerate(all_pairs):
        actual = raw_matrix.loc[gi, gj]
        shuffled = null_matrix[:, k]
        p_plus = np.mean(shuffled >= actual)
        p_minus = np.mean(shuffled <= actual)
        p_value.loc[gi, gj] = min(2 * p_plus, 2 * p_minus, 1.0)

    return raw_matrix, p_value


## 3. Time-resolved sweep (the expensive cells)

- **Twin correlation + z-score** only depends on a single timepoint, so it's computed
  once per (scenario, replicate, t) across the *entire* `ALL_TIME_STEPS` grid.
- **Directed cross-correlation** genuinely depends on the (t1, t2) pair, so it's
  computed for every t1 in `T1_ANCHORS` swept against every t2 in `ALL_TIME_STEPS`
  (step size `T2_SWEEP_STEP`). `t1 = 1h, t2 = 20h` (the originally requested pair)
  falls out of this sweep automatically and is pulled out for the headline heatmaps
  in Section 4.

Both results tables are cached to `oscillation_effect_results/*.csv` so later cells
(and re-runs that only touch plotting) don't need to redo the permutation tests.


In [ ]:
zscore_records = []
for scenario, reps in DATA.items():
    for rep_label, df in reps.items():
        for t in tqdm(ALL_TIME_STEPS, desc=f"{scenario}/{rep_label} twin corr + z-score"):
            df_t = df[df["time_step"] == t]
            twin_corr, random_point_corr, z_score = compute_twin_correlation_and_zscore(
                df_t, GENE_LIST, n_shuffles=N_SHUFFLES, base_seed=BASE_SEED
            )
            for gi, gj in combinations(GENE_LIST, 2):
                zscore_records.append(dict(
                    scenario=scenario, rep=rep_label, time_step=t, gene_i=gi, gene_j=gj,
                    twin_corr=twin_corr.loc[gi, gj],
                    random_point_corr=random_point_corr.loc[gi, gj],
                    z_score=z_score.loc[gi, gj],
                ))

zscore_df = pd.DataFrame(zscore_records)
zscore_df.to_csv(RESULTS_DIR / "twin_correlation_zscore_by_time.csv", index=False)
zscore_df.head()


In [ ]:
t2_sweep = [15] #ALL_TIME_STEPS[::T2_SWEEP_STEP]
# if ALL_TIME_STEPS[-1] not in t2_sweep:
#     t2_sweep = t2_sweep + [ALL_TIME_STEPS[-1]]
# if HEADLINE_T2 not in t2_sweep:
#     t2_sweep = sorted(set(t2_sweep) | {HEADLINE_T2})

cross_corr_records = []
for scenario, reps in DATA.items():
    for rep_label, df in reps.items():
        for t1 in T1_ANCHORS:
            df_t1 = df[df["time_step"] == t1]
            for t2 in tqdm(t2_sweep, desc=f"{scenario}/{rep_label} t1={t1}h cross-corr"):
                df_t2 = df[df["time_step"] == t2]
                raw_matrix, p_value = compute_directed_cross_correlation(
                    df_t1, df_t2, GENE_LIST, n_shuffles=N_SHUFFLES, base_seed=BASE_SEED
                )
                for gi in GENE_LIST:
                    for gj in GENE_LIST:
                        cross_corr_records.append(dict(
                            scenario=scenario, rep=rep_label, t1=t1, t2=t2, gene_i=gi, gene_j=gj,
                            cross_corr=raw_matrix.loc[gi, gj],
                            p_value=p_value.loc[gi, gj],
                        ))

cross_corr_df = pd.DataFrame(cross_corr_records)
cross_corr_df.to_csv(RESULTS_DIR / "cross_correlation_by_time.csv", index=False)
cross_corr_df.head()


## 4. Headline snapshot: t1 = 1h, t2 = 20h

All 8 metrics per scenario, at the originally requested (t1, t2) pair: twin-pair
correlation, random-pair correlation, and z-score at t1 and at t2 (6 heatmaps), plus
the directed cross-correlation and its permutation p-value from t1 to t2 (2
heatmaps). Each heatmap averages the gene x gene matrix across the 3 replicates
(per-replicate values are still available in `zscore_df` / `cross_corr_df`).


In [ ]:
def _matrix_from_long(df, value_col, row_col, col_col, gene_list, extra_query, has_diagonal):
    '''Pivot a long-format (row_col, col_col, value_col) table into a gene x gene matrix.

    has_diagonal=False is for zscore_df's twin_corr/random_point_corr/z_score, which
    only stores gene_i<gene_j pairs (undirected): the symmetric half is filled by
    reflection and the diagonal is set explicitly (1.0 for the two correlation
    columns -- a gene's delta correlated with itself is always 1, twin or random
    pairing alike -- NaN for z_score, since none of the three store a diagonal).
    has_diagonal=True is for cross_corr_df's cross_corr/p_value, which is directed
    and already stores the full gene_i x gene_j grid including the diagonal
    (self-gene autocorrelation) -- no reflection or diagonal override needed there.
    '''
    sub = df.query(extra_query)
    piv = sub.groupby([row_col, col_col])[value_col].mean().reset_index()
    mat = pd.DataFrame(np.nan, index=gene_list, columns=gene_list, dtype=float)
    for _, r in piv.iterrows():
        mat.loc[r[row_col], r[col_col]] = r[value_col]
    if not has_diagonal:
        mat = mat.where(~mat.isna(), mat.T)
        diag_value = 1.0 if value_col in ("twin_corr", "random_point_corr") else np.nan
        np.fill_diagonal(mat.values, diag_value)
    return mat


for scenario in SCENARIOS:
    twin_corr_t1 = _matrix_from_long(zscore_df, "twin_corr", "gene_i", "gene_j", GENE_LIST,
                                      "scenario == @scenario and time_step == @HEADLINE_T1", has_diagonal=False)
    twin_corr_t2 = _matrix_from_long(zscore_df, "twin_corr", "gene_i", "gene_j", GENE_LIST,
                                      "scenario == @scenario and time_step == @HEADLINE_T2", has_diagonal=False)
    z_t1 = _matrix_from_long(zscore_df, "z_score", "gene_i", "gene_j", GENE_LIST,
                              "scenario == @scenario and time_step == @HEADLINE_T1", has_diagonal=False)
    z_t2 = _matrix_from_long(zscore_df, "z_score", "gene_i", "gene_j", GENE_LIST,
                              "scenario == @scenario and time_step == @HEADLINE_T2", has_diagonal=False)
    random_corr_t1 = _matrix_from_long(zscore_df, "random_point_corr", "gene_i", "gene_j", GENE_LIST,
                                        "scenario == @scenario and time_step == @HEADLINE_T1", has_diagonal=False)
    random_corr_t2 = _matrix_from_long(zscore_df, "random_point_corr", "gene_i", "gene_j", GENE_LIST,
                                        "scenario == @scenario and time_step == @HEADLINE_T2", has_diagonal=False)
    cross_12 = _matrix_from_long(cross_corr_df, "cross_corr", "gene_i", "gene_j", GENE_LIST,
                                  "scenario == @scenario and t1 == @HEADLINE_T1 and t2 == @HEADLINE_T2", has_diagonal=True)
    cross_p_12 = _matrix_from_long(cross_corr_df, "p_value", "gene_i", "gene_j", GENE_LIST,
                                    "scenario == @scenario and t1 == @HEADLINE_T1 and t2 == @HEADLINE_T2", has_diagonal=True)

    cf.plot_matrix_as_heatmap(twin_corr_t1, GENE_LIST, title=f"[{scenario}] Twin-pair correlation",
                               add_time=True, time=[HEADLINE_T1], black_out_self=True, vmin = -0.05, vmax = 0.05)
    cf.plot_matrix_as_heatmap(twin_corr_t2, GENE_LIST, title=f"[{scenario}] Twin-pair correlation",
                               add_time=True, time=[HEADLINE_T2], black_out_self=True, vmin = -0.05, vmax = 0.05)
    cf.plot_matrix_as_heatmap(z_t1, GENE_LIST, title=f"[{scenario}] Twin vs random-pair z-score",
                               add_time=True, time=[HEADLINE_T1], black_out_self=True, vmin = -7, vmax = 5)
    cf.plot_matrix_as_heatmap(z_t2, GENE_LIST, title=f"[{scenario}] Twin vs random-pair z-score",
                               add_time=True, time=[HEADLINE_T2], black_out_self=True, vmin = -7, vmax = 5)
    cf.plot_matrix_as_heatmap(random_corr_t1, GENE_LIST, title=f"[{scenario}] Random-pair correlation",
                               add_time=True, time=[HEADLINE_T1], black_out_self=True, vmin = -0.2, vmax = 0.2)
    cf.plot_matrix_as_heatmap(random_corr_t2, GENE_LIST, title=f"[{scenario}] Random-pair correlation",
                               add_time=True, time=[HEADLINE_T2], black_out_self=True, vmin = -0.2, vmax = 0.2)
    cf.plot_matrix_as_heatmap(cross_12, GENE_LIST, title=f"[{scenario}] Directed cross-correlation", vmin = -0.2, vmax = 0.2,
                               add_time=True, time=[HEADLINE_T1, HEADLINE_T2], symmetric=False)
    cf.plot_matrix_as_heatmap(cross_p_12, GENE_LIST, title=f"[{scenario}] Cross-correlation p-value",
                               add_time=True, time=[HEADLINE_T1, HEADLINE_T2], symmetric=False,
                               vmin=0, vmax=1, cmap="viridis_r")


## 5. Twin correlation & z-score vs time

One panel per gene pair; each line is a scenario, averaged across the 3
replicates (shaded band = SD across replicates).


In [ ]:
_gene_pairs = list(combinations(GENE_LIST, 2))
_ncols = 3
_nrows = int(np.ceil(len(_gene_pairs) / _ncols))

for value_col, ylabel in [("twin_corr", "twin-pair correlation"), ("z_score", "z-score (twin vs random null)")]:
    fig, axes = plt.subplots(_nrows, _ncols, figsize=(4.2 * _ncols, 3.4 * _nrows), sharex=True)
    for ax, (gi, gj) in zip(np.ravel(axes), _gene_pairs):
        for scenario in SCENARIOS:
            sub = zscore_df.query("scenario == @scenario and gene_i == @gi and gene_j == @gj")
            agg = sub.groupby("time_step")[value_col].agg(["mean", "std"]).reset_index()
            color = _scenario_colors[scenario]
            ax.plot(agg["time_step"], agg["mean"], label=scenario, color=color, linewidth=1.5)
            ax.fill_between(agg["time_step"], agg["mean"] - agg["std"], agg["mean"] + agg["std"],
                             color=color, alpha=0.2, linewidth=0)
        ax.axhline(0, color="black", linewidth=0.6, linestyle=":")
        ax.set_title(f"{gi} – {gj}")
        ax.set_xlabel("time after division (h)")
        ax.set_ylabel(ylabel)
    for ax in np.ravel(axes)[len(_gene_pairs):]:
        ax.axis("off")
    np.ravel(axes)[0].legend(fontsize=8)
    fig.suptitle(f"{ylabel} vs time, by scenario (mean ± SD across 3 replicates)")
    plt.tight_layout()
    plt.show()


## 6. Directed cross-correlation vs t2, for each t1 anchor

One figure per scenario; one panel per `t1` anchor; each line is a directed gene
pair (solid = cross-gene, dashed = self/autocorrelation), averaged across the 3
replicates.


In [ ]:
_directed_pairs = [(gi, gj) for gi in GENE_LIST for gj in GENE_LIST]
_pair_colors = dict(zip(_directed_pairs, plt.cm.tab20(np.linspace(0, 1, len(_directed_pairs)))))

for scenario in SCENARIOS:
    fig, axes = plt.subplots(1, len(T1_ANCHORS), figsize=(4.2 * len(T1_ANCHORS), 4), sharey=True)
    for ax, t1 in zip(np.ravel(axes), T1_ANCHORS):
        for gi, gj in _directed_pairs:
            sub = cross_corr_df.query(
                "scenario == @scenario and t1 == @t1 and gene_i == @gi and gene_j == @gj"
            )
            agg = sub.groupby("t2")["cross_corr"].mean().reset_index().sort_values("t2")
            ax.plot(agg["t2"], agg["cross_corr"],
                    color=_pair_colors[(gi, gj)],
                    linestyle="--" if gi == gj else "-",
                    linewidth=1.3, label=f"{gi}→{gj}")
        ax.axvline(t1, color="black", linewidth=0.6, linestyle=":")
        ax.axhline(0, color="black", linewidth=0.6, linestyle=":")
        ax.set_title(f"t1 = {t1}h")
        ax.set_xlabel("t2 (h)")
    axes.set_ylabel("directed cross-correlation")
    fig.suptitle(f"[{scenario}] cross-correlation vs t2 (mean across 3 replicates)")
    fig.legend(*axes.get_legend_handles_labels(), loc="center left",
               bbox_to_anchor=(1.0, 0.5), fontsize=8, ncol=1)
    plt.tight_layout()
    plt.show()


## 7. Combined view: gene-gene, twin, cross-correlation & z-score vs time (out-of-phase only)

One figure, faceted by gene pair (2x3 grid), out-of-phase scenario only. Each panel
overlays four metrics on a shared time axis, all averaged across the 3 replicates:

- **Gene-gene correlation** (`calculate_pairwise_gene_gene_correlation_matrix`) —
  plain Spearman correlation of gene_i vs gene_j mRNA across all cells (no twin
  pairing), at every timepoint.
- **Twin-pair correlation** (`twin_corr` from `zscore_df`, already computed in
  Section 5 for the full time grid).
- **Cross-correlation** (`get_cross_correlations`) — directed correlation from
  gene_i at a fixed source time t1 to gene_j at every t2 on the same grid. No
  permutation p-value is computed here (not needed for this plot), so this sweep
  is cheap.
- **Z-score** (`z_score` from `zscore_df`, already computed in Section 5).

Correlation-scale metrics (gene-gene, twin-pair, cross) share the left y-axis;
z-score gets its own right y-axis since its scale differs.


In [ ]:
def compute_directed_cross_correlation_raw(df_t1, df_t2, gene_list):
    '''Directed t1->t2 cross-correlation matrix only (no permutation p-value).'''
    rep_0_t1, _ = cf.split_twin_pairs(df_t1)
    _, rep_1_t2 = cf.split_twin_pairs(df_t2)
    gene_pairs = list(permutations(gene_list, 2))
    return cf.get_cross_correlations(rep_0_t1, rep_1_t2, gene_pairs, type_comparison="twin")



In [ ]:

COMBINED_SCENARIO = "out_of_phase"
CROSS_CORR_T1 = 1  # hours; source timepoint for the cross-correlation curve

gene_gene_corr_records = []
cross_corr_sweep_records = []
for rep_label, df in DATA[COMBINED_SCENARIO].items():
    df_t1 = df[df["time_step"] == CROSS_CORR_T1]
    for t in tqdm(ALL_TIME_STEPS, desc=f"{COMBINED_SCENARIO}/{rep_label} gene-gene + cross-corr sweep"):
        df_t = df[df["time_step"] == t]

        gg_corr = cf.calculate_pairwise_gene_gene_correlation_matrix(df_t, GENE_LIST)
        for gi, gj in _gene_pairs:
            gene_gene_corr_records.append(dict(
                rep=rep_label, time_step=t, gene_i=gi, gene_j=gj,
                gene_gene_corr=gg_corr.loc[gi, gj],
            ))

        raw_matrix = compute_directed_cross_correlation_raw(df_t1, df_t, GENE_LIST)
        for gi, gj in _gene_pairs:
            cross_corr_sweep_records.append(dict(
                rep=rep_label, t1=CROSS_CORR_T1, t2=t, gene_i=gi, gene_j=gj,
                cross_corr=raw_matrix.loc[gi, gj],
            ))

gene_gene_corr_df = pd.DataFrame(gene_gene_corr_records)
cross_corr_sweep_df = pd.DataFrame(cross_corr_sweep_records)
gene_gene_corr_df.to_csv(RESULTS_DIR / f"gene_gene_correlation_{COMBINED_SCENARIO}.csv", index=False)
cross_corr_sweep_df.to_csv(RESULTS_DIR / f"cross_correlation_sweep_{COMBINED_SCENARIO}_t1_{CROSS_CORR_T1}.csv", index=False)
cross_corr_sweep_df.head()


In [ ]:
from twinfer.utils.paths import get_data_root as _twinfer_get_data_root
TWINFER_PROJECT_ROOT = _twinfer_get_data_root().parent  # [2026-09-30 added: replaces hardcoded project-root paths (/home/gzu5140/TwINFER_KA, /gpfs/projects/b1255/hzhang/TwINFER_KA, old Keerthana_b1042 tree)]
RESULTS_DIR = Path(f'{TWINFER_PROJECT_ROOT}/clean_data/notebooks/real_data_analysis/oscillation_effect_results_redo')
COMBINED_SCENARIO= "out_of_phase"
gene_gene_corr_df = pd.read_csv(RESULTS_DIR / f"gene_gene_correlation_{COMBINED_SCENARIO}.csv")
cross_corr_sweep_df = pd.read_csv(RESULTS_DIR / f"cross_correlation_sweep_{COMBINED_SCENARIO}_t1_1.csv")
zscore_df = pd.read_csv(RESULTS_DIR / "twin_correlation_zscore_by_time.csv")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
for ax, (gi, gj) in zip(axes.flat, _gene_pairs):
    sub_z = zscore_df.query("scenario == @COMBINED_SCENARIO and gene_i == @gi and gene_j == @gj")
    agg_twin = sub_z.groupby("time_step")["twin_corr"].agg(["mean", "std"]).reset_index()
    agg_z = sub_z.groupby("time_step")["z_score"].agg(["mean", "std"]).reset_index()

    agg_gg = gene_gene_corr_df.query("gene_i == @gi and gene_j == @gj") \
        .groupby("time_step")["gene_gene_corr"].agg(["mean", "std"]).reset_index()

    agg_cc = cross_corr_sweep_df.query("gene_i == @gi and gene_j == @gj") \
        .groupby("t2")["cross_corr"].agg(["mean", "std"]).reset_index().sort_values("t2")

    ax2 = ax.twinx()
    l1, = ax.plot(agg_gg["time_step"], agg_gg["mean"], color="tab:purple",
                   linewidth=1.4, label="gene-gene correlation")
    ax.fill_between(agg_gg["time_step"], agg_gg["mean"] - agg_gg["std"], agg_gg["mean"] + agg_gg["std"],
                     color="tab:purple", alpha=0.15, linewidth=0)
    l2, = ax.plot(agg_twin["time_step"], agg_twin["mean"], color="tab:blue",
                   linewidth=1.4, label="twin-pair correlation")
    ax.fill_between(agg_twin["time_step"], agg_twin["mean"] - agg_twin["std"], agg_twin["mean"] + agg_twin["std"],
                     color="tab:blue", alpha=0.15, linewidth=0)
    l3, = ax.plot(agg_cc["t2"], agg_cc["mean"], color="tab:green",
                   linewidth=1.4, label=f"cross-correlation (t1={CROSS_CORR_T1}h)")
    ax.fill_between(agg_cc["t2"], agg_cc["mean"] - agg_cc["std"], agg_cc["mean"] + agg_cc["std"],
                     color="tab:green", alpha=0.15, linewidth=0)
    l4, = ax2.plot(agg_z["time_step"], agg_z["mean"], color="tab:red",
                    linewidth=1.4, linestyle="--", label="z-score")
    ax2.fill_between(agg_z["time_step"], agg_z["mean"] - agg_z["std"], agg_z["mean"] + agg_z["std"],
                      color="tab:red", alpha=0.1, linewidth=0)

    ax.axhline(0, color="black", linewidth=0.6, linestyle=":")
    ax.axvline(CROSS_CORR_T1, color="gray", linewidth=0.8, linestyle=":")
    ax.set_title(f"{gi} \u2013 {gj}")
    ax.set_xlabel("time after division (h)")
    ax.set_ylabel("correlation")
    ax2.set_ylabel("z-score", color="tab:red")
    ax2.tick_params(axis="y", colors="tab:red")

fig.legend(handles=[l1, l2, l3, l4], loc="upper center", bbox_to_anchor=(0.5, 1.05), ncol=4, fontsize=9)
fig.suptitle(f"[{COMBINED_SCENARIO}] gene-gene, twin-pair, and cross-correlation (t1={CROSS_CORR_T1}h) "
             f"+ z-score vs time (mean \u00b1 SD across 3 replicates)", y=1.1)
plt.tight_layout()
plt.show()


## 8. figure_2 data: A_B vs A_to_B, low_k_on merged with high_k_on

Analyzes `simulation_data/figure_2/{A_B,A_to_B}_{low,high}_k_on/`. For each of the two
network topologies (`A_B`, `A_to_B`), the `low_k_on` and `high_k_on` simulations are
merged clone-by-clone using `split_and_merge_simulations` from
[`correlation_analysis_helpers.py`](../TwINFER_function_scripts/correlation_analysis_helpers.py):
for replicate index `i`, `rep_i` of `low_k_on` and `rep_i` of `high_k_on` are merged by
splitting the 6000 clone IDs evenly between the two files (clones 0-2999 from `low_k_on`,
3000-5999 from `high_k_on`), giving one merged "replicate" per index that mixes both
`k_on` regimes. This is repeated per-replicate to build `FIG2_N_REPS` merged replicates
per scenario.

Only 2 genes (`gene_1`, `gene_2`) are present in this dataset, vs. 4 in the oscillator
data above, but the same helper functions from Section 2 (`compute_twin_correlation_and_zscore`,
`compute_directed_cross_correlation_raw` from Section 7) are reused as-is.

**Compute budget:** each `figure_2` folder has 20 replicates and a 49-point (0-48h) time
grid, ~6-7x the oscillator data's 3 replicates. To keep the z-score permutation sweep
tractable, only the first `FIG2_N_REPS` merged replicates and every `FIG2_TIME_STEP_STRIDE`-th
timepoint are used. The cross-correlation sweep uses the raw (no-permutation-p-value)
helper from Section 7, since the line plot only needs the mean directed correlation.

Results are saved to `RESULTS_DIR` as `fig_2_flowchart_comp_test_*.csv`.


In [ ]:
from twinfer.utils.paths import get_data_root as _twinfer_get_data_root
TWINFER_PROJECT_ROOT = _twinfer_get_data_root().parent  # [2026-09-30 added: replaces hardcoded project-root paths (/home/gzu5140/TwINFER_KA, /gpfs/projects/b1255/hzhang/TwINFER_KA, old Keerthana_b1042 tree)]
# ---- Config: figure_2 (A_B / A_to_B, low_k_on merged with high_k_on) ---
import re

CAH_ROOT = f'{TWINFER_PROJECT_ROOT}/code/TwINFER/TwINFER_function_scripts'
if CAH_ROOT not in sys.path:
    pass
    # [2026-09-30 commented out: modules are imported via dotted package paths (env.sh puts clean_code and clean_code/package on PYTHONPATH)]
    # sys.path.insert(0, CAH_ROOT)
from correlation_analysis_helpers import split_and_merge_simulations

FIG2_BASE_DIR = Path(f'{TWINFER_PROJECT_ROOT}/simulation_data/figure_2')
FIG2_SCENARIOS = ["A_B", "A_to_B"]   # each merges "<scenario>_low_k_on" + "<scenario>_high_k_on"

FIG2_N_REPS = 5             # use only the first N merged replicates (out of 20 available)
FIG2_TIME_STEP_STRIDE = 2   # subsample every Nth timepoint from the full 0..48h grid

FIG2_T1_ANCHORS = [5]       # hours; cross-correlation reference (source) timepoints

FIG2_RESULTS_PREFIX = "fig_2_flowchart_comp_test"


In [ ]:
# ---- Discover & pair low_k_on / high_k_on replicate files ----------------
def discover_fig2_replicate_files(folder):
    '''df_rows_*.csv files in a figure_2 subfolder, keyed by rep index (e.g. "rep_0").'''
    reps = {}
    for p in sorted(folder.glob("df_rows_*.csv")):
        m = re.search(r"_rep_(\d+)\.csv$", p.name)
        reps[f"rep_{int(m.group(1))}"] = p
    return dict(sorted(reps.items(), key=lambda kv: int(kv[0].split("_")[1])))


FIG2_FILES = {}
for scenario in FIG2_SCENARIOS:
    low_reps = discover_fig2_replicate_files(FIG2_BASE_DIR / f"{scenario}_low_k_on")
    high_reps = discover_fig2_replicate_files(FIG2_BASE_DIR / f"{scenario}_high_k_on")
    assert set(low_reps) == set(high_reps), f"{scenario}: low/high_k_on rep labels differ"
    rep_labels = list(low_reps)[:FIG2_N_REPS]
    FIG2_FILES[scenario] = {rep: (low_reps[rep], high_reps[rep]) for rep in rep_labels}

for scenario, reps in FIG2_FILES.items():
    print(scenario, "-> merging", list(reps.keys()))


In [ ]:
# ---- Merge low_k_on + high_k_on per replicate, load, sanity-check --------
FIG2_DATA = {
    scenario: {
        rep_label: split_and_merge_simulations([str(low_path), str(high_path)])
        for rep_label, (low_path, high_path) in reps.items()
    }
    for scenario, reps in FIG2_FILES.items()
}

_fig2_first_df = next(iter(next(iter(FIG2_DATA.values())).values()))
FIG2_GENE_LIST_RAW = sorted({c.rsplit("_", 1)[0] for c in _fig2_first_df.columns if c.endswith("_mRNA")})
FIG2_GENE_LIST, _ = cf.sort_genes_numerically(FIG2_GENE_LIST_RAW)
print("genes:", FIG2_GENE_LIST)

_fig2_full_time_steps = sorted(_fig2_first_df["time_step"].unique().tolist())
for scenario, reps in FIG2_DATA.items():
    for rep_label, df in reps.items():
        ts = sorted(df["time_step"].unique().tolist())
        assert ts == _fig2_full_time_steps, f"{scenario}/{rep_label} has a different time grid: {ts}"

FIG2_ALL_TIME_STEPS = _fig2_full_time_steps[::FIG2_TIME_STEP_STRIDE]
if _fig2_full_time_steps[-1] not in FIG2_ALL_TIME_STEPS:
    FIG2_ALL_TIME_STEPS.append(_fig2_full_time_steps[-1])
print(f"time_step grid (every {FIG2_TIME_STEP_STRIDE}h): {FIG2_ALL_TIME_STEPS}")

_sample_scenario, _sample_reps = next(iter(FIG2_DATA.items()))
_sample_rep_label, _sample_df = next(iter(_sample_reps.items()))
print(f"sanity check: merged {_sample_scenario}/{_sample_rep_label} has "
      f"{_sample_df['clone_id'].nunique()} unique clone_ids ({_sample_df.shape[0]} rows)")


### 8a. Twin-pair correlation, z-score, and cross-correlation sweeps

Reuses `compute_twin_correlation_and_zscore` (Section 2) and `compute_directed_cross_correlation_raw`
(Section 7, no permutation p-value needed for the line plot) unchanged, just pointed at
`FIG2_DATA` / `FIG2_GENE_LIST` / `FIG2_ALL_TIME_STEPS`.


In [ ]:
fig2_zscore_records = []
for scenario, reps in FIG2_DATA.items():
    for rep_label, df in reps.items():
        for t in tqdm(FIG2_ALL_TIME_STEPS, desc=f"{scenario}/{rep_label} twin corr + z-score"):
            df_t = df[df["time_step"] == t]
            twin_corr, random_point_corr, z_score = compute_twin_correlation_and_zscore(
                df_t, FIG2_GENE_LIST, n_shuffles=N_SHUFFLES, base_seed=BASE_SEED
            )
            for gi, gj in combinations(FIG2_GENE_LIST, 2):
                fig2_zscore_records.append(dict(
                    scenario=scenario, rep=rep_label, time_step=t, gene_i=gi, gene_j=gj,
                    twin_corr=twin_corr.loc[gi, gj],
                    random_point_corr=random_point_corr.loc[gi, gj],
                    z_score=z_score.loc[gi, gj],
                ))

fig2_zscore_df = pd.DataFrame(fig2_zscore_records)
fig2_zscore_df.to_csv(RESULTS_DIR / f"{FIG2_RESULTS_PREFIX}_twin_correlation_zscore_by_time.csv", index=False)
fig2_zscore_df.head()


In [ ]:
fig2_cross_corr_records = []
for scenario, reps in FIG2_DATA.items():
    for rep_label, df in reps.items():
        for t1 in [1]:
            df_t1 = df[df["time_step"] == t1]
            for t2 in tqdm(FIG2_ALL_TIME_STEPS, desc=f"{scenario}/{rep_label} t1={t1}h cross-corr"):
                df_t2 = df[df["time_step"] == t2]
                raw_matrix = compute_directed_cross_correlation_raw(df_t1, df_t2, FIG2_GENE_LIST)
                for gi in FIG2_GENE_LIST:
                    for gj in FIG2_GENE_LIST:
                        fig2_cross_corr_records.append(dict(
                            scenario=scenario, rep=rep_label, t1=t1, t2=t2, gene_i=gi, gene_j=gj,
                            cross_corr=raw_matrix.loc[gi, gj],
                        ))

fig2_cross_corr_df = pd.DataFrame(fig2_cross_corr_records)
fig2_cross_corr_df.to_csv(RESULTS_DIR / f"{FIG2_RESULTS_PREFIX}_cross_correlation_by_time.csv", index=False)
fig2_cross_corr_df.head()

### 8b. Plots

Both figures below use `sharex=True, sharey=True` across every subplot so axis ranges
are directly comparable panel-to-panel.


In [ ]:
fig2_zscore_df = pd.read_csv(RESULTS_DIR / f"{FIG2_RESULTS_PREFIX}_twin_correlation_zscore_by_time.csv")
fig2_cross_corr_df = pd.read_csv(RESULTS_DIR / f"{FIG2_RESULTS_PREFIX}_cross_correlation_by_time.csv")

In [ ]:
_fig2_gene_pairs = list(combinations(FIG2_GENE_LIST, 2))
_fig2_scenario_colors = dict(zip(FIG2_SCENARIOS, plt.cm.tab10.colors))
_ncols = 3
_nrows = int(np.ceil(len(_fig2_gene_pairs) / _ncols))

for value_col, ylabel in [("twin_corr", "twin-pair correlation"), ("z_score", "z-score (twin vs random null)")]:
    fig, axes = plt.subplots(_nrows, _ncols, figsize=(4.2 * _ncols, 3.4 * _nrows),
                              sharex=True, sharey=True, squeeze=False)
    for ax, (gi, gj) in zip(np.ravel(axes), _fig2_gene_pairs):
        for scenario in FIG2_SCENARIOS:
            sub = fig2_zscore_df.query("scenario == @scenario and gene_i == @gi and gene_j == @gj")
            agg = sub.groupby("time_step")[value_col].agg(["mean", "std"]).reset_index()
            color = _fig2_scenario_colors[scenario]
            ax.plot(agg["time_step"], agg["mean"], label=scenario, color=color, linewidth=1.5)
            ax.fill_between(agg["time_step"], agg["mean"] - agg["std"], agg["mean"] + agg["std"],
                             color=color, alpha=0.2, linewidth=0)
        ax.axhline(0, color="black", linewidth=0.6, linestyle=":")
        ax.set_title(f"{gi} \u2013 {gj}")
        ax.set_xlabel("time after division (h)")
        ax.set_ylabel(ylabel)
    for ax in np.ravel(axes)[len(_fig2_gene_pairs):]:
        ax.axis("off")
    np.ravel(axes)[0].legend(fontsize=8)
    fig.suptitle(f"[figure_2: A_B vs A_to_B, low+high k_on merged] {ylabel} vs time "
                 f"(mean \u00b1 SD across {FIG2_N_REPS} merged replicates)")
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
FIG2_GENE_LIST = ['gene_1', 'gene_2']
_fig2_directed_pairs = [(gi, gj) for gi in FIG2_GENE_LIST for gj in FIG2_GENE_LIST]
_fig2_pair_colors = dict(zip(_fig2_directed_pairs, plt.cm.tab20(np.linspace(0, 1, len(_fig2_directed_pairs)))))

fig, axes = plt.subplots(len(FIG2_T1_ANCHORS), len(FIG2_SCENARIOS),
                          figsize=(4.5 * len(FIG2_SCENARIOS), 3.6 * len(FIG2_T1_ANCHORS)),
                          sharex=True, sharey=True, squeeze=False)
for row, t1 in enumerate([1]):
    for col, scenario in enumerate(FIG2_SCENARIOS):
        ax = axes[row, col]
        for gi, gj in _fig2_directed_pairs:
            sub = fig2_cross_corr_df.query(
                "scenario == @scenario and t1 == @t1 and gene_i == @gi and gene_j == @gj"
            )
            agg = sub.groupby("t2")["cross_corr"].mean().reset_index().sort_values("t2")
            ax.plot(agg["t2"], agg["cross_corr"],
                    color=_fig2_pair_colors[(gi, gj)],
                    linestyle="--" if gi == gj else "-",
                    linewidth=1.3, label=f"{gi}\u2192{gj}")
        ax.axvline(t1, color="black", linewidth=0.6, linestyle=":")
        ax.axhline(0, color="black", linewidth=0.6, linestyle=":")
        ax.set_title(f"[{scenario}] t1 = {t1}h")
        ax.set_xlabel("t2 (h)")

axes[0, 0].set_ylabel("directed cross-correlation")
fig.suptitle(f"[figure_2: low+high k_on merged] cross-correlation vs t2 "
             f"(mean across {FIG2_N_REPS} merged replicates)")
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="center left",
           bbox_to_anchor=(1.0, 0.5), fontsize=8, ncol=1)
plt.tight_layout()
plt.show()
